# Traffic CCTV — UVH-26 YOLO11-X, batch inference trên 5 camera (**chỉ ảnh all_boxes**)

**Giữ nguyên UVH-26 YOLO11-X (`uvh26_x` mặc định), kích thước inference, confidence, NMS, và màu bounding box của notebook `detect-n.ipynb`.** Mỗi frame hợp lệ chỉ sinh **một ảnh có tất cả bounding boxes**, không sinh `two_wheelers_only`, `crowded_zoom`, hay CSV/JSON *riêng từng frame*.

Đọc tất cả ảnh `frame_*` trong **đúng 5 thư mục camera**; bỏ qua `summary.json`, `metadata.csv` và file không phải ảnh. Giữ cấu trúc 5 thư mục trong ZIP. Có thêm các file **tổng hợp cấp dataset**: `manifest.csv`, `camera_summary.csv`, `errors.csv`, `run_config.json` (không tính là output riêng của mỗi ảnh).

**Chỉ inference, không train/tracking.** Count trong `manifest.csv` là số detection theo frame, **không** suy ra traffic flow/OD/density thật hoặc độ chính xác khi chưa có ground truth.

**Kaggle:** Add Input dataset → bật GPU → bật Internet (nếu cần tải checkpoint) → Run All → tải ZIP tại `/kaggle/working/traffic_detection_BMD45_5cams_all_boxes_only.zip`.

In [ ]:
%pip install -q -U ultralytics huggingface_hub pillow pandas matplotlib

## 1) Cấu hình

`INPUT_ROOT` là thư mục cha của 5 thư mục camera; notebook có fallback tìm trong `/kaggle/input`.

Mặc định `OUTPUT_IMAGE_FORMAT="png"` giữ chất lượng ảnh. Có thể đổi sang `"jpg"` để giảm dung lượng ZIP mà **không ảnh hưởng kết quả inference**, chỉ thay đổi chất lượng lưu ảnh. `MAX_FRAMES_PER_CAMERA=None` chạy toàn bộ. Không có tùy chọn vẽ ảnh riêng cho xe hai bánh hoặc crowded zoom.

In [ ]:
INPUT_ROOT = "/kaggle/input/datasets/rykanny/traffic-benchmark/TEST_traffic_camera_5_locations"
CAMERA_FOLDERS = [
    "Cam_Ly_Thuong_Kiet_Bac_Hai",
    "Cam_Thanh_Thai_To_Hien_Thanh",
    "Cam_To_Hien_Thanh_1",
    "Cam_To_Hien_Thanh_2",
    "Cam_To_Hien_Thanh_Su_Van_Hanh",
]

# Giữ nguyên pipeline YOLO, chỉ chọn checkpoint UVH-26 đã fine-tune.
MODEL_CHOICE = "uvh26_x"      # Checkpoint UVH-26-MV-YOLOv11-X.pt
PAPER_WEIGHTS_PATH = "/kaggle/input/paper-author-weights/best.pt"
CUSTOM_WEIGHTS_PATH = "/kaggle/input/my-checkpoint/best.pt"

# Inference giữ NGUYÊN như detect-n.ipynb
IMAGE_SIZE = 960
CONFIDENCE = 0.12
NMS_IOU = 0.65
MAX_DETECTIONS = 600
TEST_TIME_AUGMENTATION = False

# Vẽ MỘT ảnh all_boxes trên mỗi frame
SHOW_CONFIDENCE_ON_BOX = False
SHOW_INDEX_ON_BOX = True
OUTPUT_IMAGE_FORMAT = "png"      # "png" hoặc "jpg"
JPEG_QUALITY = 92

# Batch, log, output ZIP; dùng tên mới để không lẫn với run cũ có 3 loại ảnh
MAX_FRAMES_PER_CAMERA = None      # None = TẤT CẢ; đặt 2 để test nhanh
PREVIEW_PER_CAMERA = 1           # 1 preview/camera (hiển thị, KHÔNG lưu ảnh phụ)
LOG_EVERY_N_FRAMES = 20
SKIP_COMPLETED_FRAMES = False    # QUAN TRỌNG: chạy mới để hiện preview, không skip 29 frame cũ
OUTPUT_DIR = "/kaggle/working/traffic_detection_UVH26_YOLO11X_FRESH_all_boxes_only"
ZIP_PATH = "/kaggle/working/traffic_detection_UVH26_YOLO11X_FRESH_5cams_all_boxes_only.zip"

## 2) Kiểm tra GPU và **quét đúng 5 thư mục** (chỉ `frame_*` là ảnh)

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime
import hashlib, json, re, time, zipfile

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import torch
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
FRAME_PREFIX = "frame_"
input_root = Path(INPUT_ROOT)
if not input_root.is_dir():
    candidates = [p for p in Path("/kaggle/input").rglob("TEST_traffic_camera_5_locations") if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(
            f"Không thấy INPUT_ROOT: {INPUT_ROOT}. Hãy Add Input dataset traffic-benchmark "
            "và kiểm tra thư mục trong thanh bên phải của Kaggle."
        )
    valid = [p for p in candidates if all((p / cam).is_dir() for cam in CAMERA_FOLDERS)]
    if len(valid) != 1:
        raise RuntimeError(f"Không thể tự chọn duy nhất dataset 5 camera. Candidates: {candidates}. Hãy sửa INPUT_ROOT.")
    input_root = valid[0]
    print(f"Đã tự tìm thấy dataset tại: {input_root}")

missing = [cam for cam in CAMERA_FOLDERS if not (input_root / cam).is_dir()]
if missing:
    raise FileNotFoundError(f"Thiếu thư mục camera: {missing}. INPUT_ROOT hiện tại: {input_root}")

# Không quét các nhánh ngoài CAMERA_FOLDERS; loại summary.json, .csv, v.v. theo extension + prefix.
camera_files = {}
for cam in CAMERA_FOLDERS:
    frames = sorted(
        p for p in (input_root / cam).rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        and p.name.lower().startswith(FRAME_PREFIX)
    )
    camera_files[cam] = frames if MAX_FRAMES_PER_CAMERA is None else frames[:MAX_FRAMES_PER_CAMERA]

inventory = pd.DataFrame([
    {"camera": cam, "frames_selected": len(camera_files[cam])}
    for cam in CAMERA_FOLDERS
])
print("INPUT_ROOT:", input_root)
display(inventory)
total_frames = int(inventory.frames_selected.sum())
print("TỔNG FRAME:", total_frames)
if total_frames == 0:
    raise RuntimeError("Không có ảnh frame_* trong 5 thư mục. Kiểm tra tên file và INPUT_ROOT.")

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU (sẽ chậm với YOLOv12-X)")
if OUTPUT_IMAGE_FORMAT.lower() not in {"png", "jpg"}:
    raise ValueError("OUTPUT_IMAGE_FORMAT phải là 'png' hoặc 'jpg'.")

## 3) Nạp **một lần** checkpoint UVH-26 YOLO11-X đã fine-tune

In [ ]:
BMD45_REPO = "iisc-aim/UVH-26"
BMD45_FILES = {
    "uvh26_x": "weights/YOLOv11-X/UVH-26-MV-YOLOv11-X.pt",
}

if MODEL_CHOICE in BMD45_FILES:
    filename = BMD45_FILES[MODEL_CHOICE]
    print(f"Đang lấy {BMD45_REPO}/{filename} (UVH-26 đã fine-tune)")
    try:
        weights_path = Path(hf_hub_download(repo_id=BMD45_REPO, filename=filename, revision="v1.0"))
    except Exception as exc:
        raise RuntimeError(
            "Không tải được checkpoint Hugging Face. Bật Internet trên Kaggle "
            "hoặc Add Input checkpoint và đặt MODEL_CHOICE='custom_local'."
        ) from exc
elif MODEL_CHOICE == "paper_local":
    weights_path = Path(PAPER_WEIGHTS_PATH)
    if not weights_path.is_file():
        raise FileNotFoundError(f"Chưa có checkpoint paper do bạn cung cấp: {weights_path}")
elif MODEL_CHOICE == "custom_local":
    weights_path = Path(CUSTOM_WEIGHTS_PATH)
    if not weights_path.is_file():
        raise FileNotFoundError(f"Không có CUSTOM_WEIGHTS_PATH: {weights_path}")
else:
    raise ValueError(f"MODEL_CHOICE chưa hỗ trợ: {MODEL_CHOICE}")

# SHA256 theo từng chunk để không nạp checkpoint weights lên RAM chỉ để hash.
sha = hashlib.sha256()
with weights_path.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        sha.update(chunk)
weights_sha256 = sha.hexdigest()
print("Checkpoint:", weights_path)
print("Size (MB):", round(weights_path.stat().st_size / (1024 * 1024), 2))
print("SHA256:", weights_sha256)
print("⚠️ Chỉ nạp checkpoint .pt từ nguồn đáng tin cậy.")

model = YOLO(str(weights_path))
class_names = {int(k): str(v) for k, v in model.names.items()} if isinstance(model.names, dict) else dict(enumerate(model.names))
print("\nClasses từ checkpoint:")
for idx, name in sorted(class_names.items()):
    print(f"  {idx:>2}: {name}")
if MODEL_CHOICE == "uvh26_x":
    assert any("two" in name.lower() and "wheeler" in name.lower() for name in class_names.values()), (
        "Checkpoint UVH-26 không có class Two-wheeler như dự kiến."
    )

# Không đoán class ID — giống notebook gốc.
TWO_WHEELER_CLASS_IDS = [
    idx for idx, name in class_names.items()
    if re.search(r"two[ -]?wheeler|motorbike|motorcycle|scooter|moped", name, re.I)
]
print("Two-wheeler class IDs:", TWO_WHEELER_CLASS_IDS)

## 4) Inference từng ảnh — giữ nguyên checkpoint và tham số của notebook gốc

In [ ]:
DETECTION_COLUMNS = [
    "index", "class_id", "class_name", "confidence", "x1", "y1", "x2", "y2",
    "cx", "cy", "width", "height", "is_two_wheeler"
]
def detect_one_frame(image_path):
    start = time.perf_counter()
    res = model.predict(
        source=str(image_path), imgsz=IMAGE_SIZE,
        conf=CONFIDENCE, iou=NMS_IOU, max_det=MAX_DETECTIONS,
        augment=TEST_TIME_AUGMENTATION, device=DEVICE,
        verbose=False, save=False
    )[0]
    elapsed = time.perf_counter() - start

    rows = []
    if res.boxes is not None:
        for idx, box in enumerate(res.boxes):
            cls_id = int(box.cls.item())
            xyxy = [float(x) for x in box.xyxy[0].tolist()]
            confidence = float(box.conf.item())
            x1, y1, x2, y2 = xyxy
            rows.append(dict(
                index=idx+1, class_id=cls_id, class_name=class_names.get(cls_id, str(cls_id)),
                confidence=round(confidence, 5),
                x1=round(x1, 2), y1=round(y1, 2), x2=round(x2, 2), y2=round(y2, 2),
                cx=round((x1+x2)/2, 2), cy=round((y1+y2)/2, 2),
                width=round(x2-x1, 2), height=round(y2-y1, 2),
                is_two_wheeler=(cls_id in TWO_WHEELER_CLASS_IDS)
            ))
    return pd.DataFrame(rows, columns=DETECTION_COLUMNS), elapsed

## 5) Vẽ và lưu **một ảnh all_boxes** (xe hai bánh màu cam, phương tiện khác màu xanh)

In [ ]:
def draw_boxes(pil_image, det_df):
    canvas = pil_image.copy()
    drawer = ImageDraw.Draw(canvas)
    thickness = max(2, round(min(pil_image.size) / 300))
    for row in det_df.itertuples(index=False):
        is_two = bool(row.is_two_wheeler)
        color = (255, 125, 15) if is_two else (25, 210, 155)
        rect = tuple(map(lambda x: int(round(x)), (row.x1, row.y1, row.x2, row.y2)))
        drawer.rectangle(rect, outline=color, width=thickness)
        tag = str(row.index) if SHOW_INDEX_ON_BOX else ""
        if SHOW_CONFIDENCE_ON_BOX:
            tag = (tag + " " if tag else "") + f"{row.confidence:.2f}"
        if tag:
            drawer.text((max(0, rect[0]), max(0, rect[1]-12)), tag, fill=color,
                        stroke_width=1, stroke_fill=(0,0,0))
    return canvas


def save_visualization(image, path):
    if OUTPUT_IMAGE_FORMAT.lower() == "jpg":
        image.save(path, format="JPEG", quality=JPEG_QUALITY, optimize=True)
    else:
        image.save(path, format="PNG", compress_level=3)


def frame_timestamp_from_name(path):
    # Chỉ trích xuất từ tên file, KHÔNG xác nhận đồng hồ camera đã đồng bộ.
    match = re.search(r"(?P<date>20\d{6})_(?P<clock>\d{6})(?:_(?P<millis>\d{1,6}))?", path.stem)
    if not match:
        return None
    try:
        t = datetime.strptime(match["date"] + match["clock"], "%Y%m%d%H%M%S")
        ms = match["millis"]
        return t.isoformat(timespec="seconds") + ("." + ms if ms else "")
    except ValueError:
        return None

## 6) Chạy 5 camera — 1 ảnh all_boxes/frame, báo cáo tổng hợp và resume

In [ ]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Đảm bảo không trộn kết quả có khác cấu hình hoặc cấu trúc output.
run_config = {
    "input_root": str(input_root), "camera_folders": CAMERA_FOLDERS,
    "frame_filter": "frame_* + extensions: .jpg,.jpeg,.png,.webp",
    "model_choice": MODEL_CHOICE, "weights_sha256": weights_sha256,
    "image_size": IMAGE_SIZE, "confidence": CONFIDENCE, "nms_iou": NMS_IOU,
    "max_detections": MAX_DETECTIONS, "test_time_augmentation": TEST_TIME_AUGMENTATION,
    "class_names": class_names, "two_wheeler_class_ids": TWO_WHEELER_CLASS_IDS,
    "show_confidence_on_box": SHOW_CONFIDENCE_ON_BOX,
    "show_index_on_box": SHOW_INDEX_ON_BOX,
    "output_image_format": OUTPUT_IMAGE_FORMAT.lower(), "jpeg_quality": JPEG_QUALITY,
    "output_contract": "one_all_boxes_image_per_frame_no_per_frame_csv_json",
}
config_path = output_dir / "run_config.json"
if config_path.exists():
    old_config = json.loads(config_path.read_text(encoding="utf-8"))
    if old_config != json.loads(json.dumps(run_config)):
        raise RuntimeError(
            f"OUTPUT_DIR đã chứa kết quả với cấu hình khác: {output_dir}. "
            "Hãy dùng OUTPUT_DIR/ZIP_PATH mới hoặc xóa output cũ để tránh trộn kết quả."
        )
else:
    if any(output_dir.iterdir()):
        raise RuntimeError(f"{output_dir} có dữ liệu nhưng thiếu run_config.json; hãy dùng OUTPUT_DIR mới.")
    config_path.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")

if OUTPUT_IMAGE_FORMAT.lower() not in ("jpg", "png"):
    raise ValueError('OUTPUT_IMAGE_FORMAT chỉ nhận "jpg" hoặc "png".')
if LOG_EVERY_N_FRAMES < 1:
    raise ValueError("LOG_EVERY_N_FRAMES phải >= 1.")


def get_output_path(image_path):
    relative = image_path.relative_to(input_root)
    # Phân biệt frame_0001.jpg và frame_0001.png nếu trùng stem.
    prefix = f"{image_path.stem}_{image_path.suffix.lstrip('.').lower()}_{MODEL_CHOICE}"
    ext = OUTPUT_IMAGE_FORMAT.lower()
    return output_dir / relative.parent / f"{prefix}_all_boxes.{ext}"


def relative_output(path):
    return str(path.relative_to(output_dir)).replace("\\", "/")


manifest_columns = [
    "camera", "frame", "input_relative_path", "timestamp_from_filename",
    "width", "height", "total_predictions", "two_wheeler_predictions",
    "inference_seconds", "status", "all_boxes_file"
]
manifest_path = output_dir / "manifest.csv"
# Resume không cần tạo metadata file riêng cho từng ảnh.
previous = {}
if manifest_path.is_file():
    prev_df = pd.read_csv(manifest_path, keep_default_na=False)
    if all(col in prev_df.columns for col in manifest_columns):
        for item in prev_df.to_dict(orient="records"):
            previous[str(item["input_relative_path"])] = item
    else:
        raise RuntimeError("manifest.csv cũ có schema khác. Hãy sử dụng OUTPUT_DIR mới.")

manifest_by_path = dict(previous)
error_rows = []
preview_counts = Counter()
newly_processed = 0
skipped = 0
t0 = time.perf_counter()


def save_manifest_checkpoint():
    # Chỉ một manifest chung, ghi atomic để có thể resume khi Kaggle bị ngắt.
    frame = pd.DataFrame(list(manifest_by_path.values()), columns=manifest_columns)
    tmp = manifest_path.with_suffix(".csv.tmp")
    frame.to_csv(tmp, index=False)
    tmp.replace(manifest_path)


for camera, frame_paths in camera_files.items():
    print(f"\n{'=' * 65}\nCAMERA: {camera} — {len(frame_paths)} frame")
    for local_index, image_path in enumerate(frame_paths, start=1):
        key = str(image_path.relative_to(input_root)).replace("\\", "/")
        out_path = get_output_path(image_path)
        try:
            prior = previous.get(key)
            if (SKIP_COMPLETED_FRAMES and prior is not None
                    and prior.get("all_boxes_file", "") == relative_output(out_path)
                    and out_path.is_file()):
                # Giữ các thống kê gốc, không inference lại.
                manifest_by_path[key] = {**prior, "status": "skipped_existing"}
                skipped += 1
                continue

            with Image.open(image_path) as source_img:
                image = source_img.convert("RGB")
            W, H = image.size
            df, elapsed = detect_one_frame(image_path)
            out_path.parent.mkdir(parents=True, exist_ok=True)

            # ĐÚNG MỘT file hình/frame: all_boxes, không zoom, không two_wheelers_only.
            all_boxes_img = draw_boxes(image, df)
            tmp_image = out_path.with_name(out_path.name + ".tmp")
            save_visualization(all_boxes_img, tmp_image)
            tmp_image.replace(out_path)

            manifest_by_path[key] = {
                "camera": camera, "frame": image_path.name,
                "input_relative_path": key,
                "timestamp_from_filename": frame_timestamp_from_name(image_path),
                "width": W, "height": H,
                "total_predictions": int(len(df)),
                "two_wheeler_predictions": int(df.is_two_wheeler.sum()),
                "inference_seconds": round(elapsed, 4),
                "status": "processed", "all_boxes_file": relative_output(out_path),
            }
            newly_processed += 1

            if preview_counts[camera] < PREVIEW_PER_CAMERA:
                preview_counts[camera] += 1
                fig, axes = plt.subplots(1, 2, figsize=(14, 5))
                axes[0].imshow(image)
                axes[0].set_title(f"{camera} — ảnh gốc")
                axes[0].axis("off")
                axes[1].imshow(all_boxes_img)
                axes[1].set_title(f"All boxes: {len(df)} | Two-wheeler: {int(df.is_two_wheeler.sum())}")
                axes[1].axis("off")
                plt.tight_layout(); plt.show(); plt.close(fig)

            if newly_processed % LOG_EVERY_N_FRAMES == 0:
                save_manifest_checkpoint()
                print(f"  Đã detect {newly_processed} frame mới, skip {skipped} | {camera}: {local_index}/{len(frame_paths)}")
        except Exception as exc:
            error_rows.append({
                "camera": camera, "input_relative_path": key,
                "error_type": type(exc).__name__, "error": str(exc)[:1000]
            })
            print(f"  ⚠️ Lỗi frame {image_path.name}: {type(exc).__name__}: {exc}")
            # Một frame hỏng không làm dừng cả batch.
    save_manifest_checkpoint()

manifest = pd.DataFrame(list(manifest_by_path.values()), columns=manifest_columns)
errors = pd.DataFrame(error_rows, columns=["camera", "input_relative_path", "error_type", "error"])
save_manifest_checkpoint()
errors.to_csv(output_dir / "errors.csv", index=False)

summary_rows = []
for camera in CAMERA_FOLDERS:
    part = manifest.loc[manifest.camera == camera]
    summary_rows.append({
        "camera": camera, "frames_discovered_this_run": len(camera_files[camera]),
        "frames_success_in_manifest": len(part),
        "frames_errors_this_run": int((errors.camera == camera).sum()),
        "sum_detections_across_frames": int(part.total_predictions.sum()),
        "sum_two_wheeler_detections_across_frames": int(part.two_wheeler_predictions.sum()),
        "avg_detections_per_successful_frame": round(float(part.total_predictions.mean()), 3) if len(part) else None,
        "avg_two_wheelers_per_successful_frame": round(float(part.two_wheeler_predictions.mean()), 3) if len(part) else None,
    })
camera_summary = pd.DataFrame(summary_rows)
camera_summary.to_csv(output_dir / "camera_summary.csv", index=False)

print("\nHOÀN THÀNH BATCH DETECTION — mỗi frame chỉ có một ảnh all_boxes")
print(f"Đã ghi {newly_processed} frame mới, skip {skipped}, lỗi trong lần chạy này: {len(errors)}")
print(f"Thời gian cell detect: {(time.perf_counter() - t0)/60:.2f} phút")
display(camera_summary)
if len(errors):
    display(errors.head(20))

## 7) Nén một ZIP — chỉ ảnh `all_boxes` và 4 file tổng hợp

In [ ]:
zip_path = Path(ZIP_PATH)
zip_path.parent.mkdir(parents=True, exist_ok=True)
if zip_path.resolve() == output_dir.resolve() or output_dir.resolve() in zip_path.resolve().parents:
    raise ValueError("ZIP_PATH phải nằm bên ngoài OUTPUT_DIR để tránh zip tự chứa chính nó.")

t0_zip = time.perf_counter()
files_to_zip = sorted(p for p in output_dir.rglob("*") if p.is_file() and not p.name.endswith(".tmp"))
with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as zf:
    for file_path in files_to_zip:
        relative_name = (Path(output_dir.name) / file_path.relative_to(output_dir)).as_posix()
        zf.write(file_path, arcname=relative_name)

# ZIP integrity check: xác minh directory index và sự hiện diện của manifest.
with zipfile.ZipFile(zip_path) as zf:
    archive_names = set(zf.namelist())
    expected = f"{output_dir.name}/manifest.csv"
    if expected not in archive_names:
        raise RuntimeError(f"ZIP thiếu {expected}.")
print("ZIP đã tạo:", zip_path)
print("Dung lượng ZIP (MB):", round(zip_path.stat().st_size / (1024 * 1024), 2))
print("Số file trong ZIP:", len(files_to_zip))
print("Thời gian nén (giây):", round(time.perf_counter() - t0_zip, 2))
print("Trên Kaggle: Output → /kaggle/working →", zip_path.name, "→ Download")
if len(errors):
    print(f"⚠️ Có {len(errors)} frame xử lý lỗi; xem errors.csv trong ZIP. ZIP vẫn được tạo với các frame thành công.")

## 8) Cấu trúc output

```text
/kaggle/working/
├── traffic_detection_UVH26_YOLO11X_FRESH_5cams_all_boxes_only.zip       ← TẢI FILE NÀY
└── traffic_detection_UVH26_YOLO11X_FRESH_all_boxes_only/
    ├── run_config.json
    ├── manifest.csv            ← 1 dòng / frame đã xử lý, kèm count và đường dẫn ảnh
    ├── camera_summary.csv      ← thống kê tổng hợp mỗi camera
    ├── errors.csv              ← lỗi của lần chạy gần nhất
    ├── Cam_Ly_Thuong_Kiet_Bac_Hai/
    │   ├── frame_000001_..._jpg_bmd_x_all_boxes.png
    │   └── frame_000002_..._jpg_bmd_x_all_boxes.png
    ├── Cam_Thanh_Thai_To_Hien_Thanh/
    ├── Cam_To_Hien_Thanh_1/
    ├── Cam_To_Hien_Thanh_2/
    └── Cam_To_Hien_Thanh_Su_Van_Hanh/
```

**Mỗi frame có đúng 1 ảnh kết quả** (`*_all_boxes.png` hoặc `.jpg`), không thêm `*_two_wheelers.*`, `*_crowded_zoom.*`, `*_detections.csv`, `*_threshold_counts.csv` hay `*_metadata.json`.

`manifest.csv` là file cấp dataset dùng ghi nhận kết quả và hỗ trợ resume, không phải file riêng theo frame. Không đưa input `summary.json`/`metadata.csv` vào ZIP. Preview trong Kaggle chỉ hiển thị, không ghi ảnh phụ.

- Chạy tất cả: `MAX_FRAMES_PER_CAMERA = None`.
- Chạy thử 2 frame/camera: `MAX_FRAMES_PER_CAMERA = 2`, sau đó đổi lại `None` để chạy tiếp.
- Thay model/threshold/cách vẽ → chọn `OUTPUT_DIR` mới để tránh trộn các run.
- Các count chỉ là số đối tượng mô hình phát hiện tại thời điểm chụp ảnh; không phải traffic flow, density chuẩn hay ground truth.